In [ ]:
pip install lib

In [ ]:
pip install --upgrade google-api-python-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.2/13.2 MB 30.2 MB/s eta 0:00:00
  Attempting uninstall: google-api-python-client
    Found existing installation: google-api-python-client 2.164.0
    Uninstalling google-api-python-client-2.164.0:
      Successfully uninstalled google-api-python-client-2.164.0


In [ ]:
pip install --upgrade google-auth-oauthlib google-auth-httplib2

In [ ]:
from googleapiclient.discovery import build
import pandas as pd
from IPython.display import JSON, display
import json

In [ ]:
# Youtube API key
api_key = 'yourAPIkey'

api_service_name = "youtube"
api_version = "v3"
# Getting credentials and create an API client
youtube = build(
    api_service_name, api_version, developerKey=api_key)


In [ ]:
channel_handles=['jayzern'] # replace one channel at a time - 5MinuteCraftsYouTube, BRIGHTSIDEOFFICIAL,buzzfeedtasty,BuzzFeedVideo,tina_yong, Thuvu5,TheFitnessMarshall,grateandgarnish5877, averysmith, VickyZhaoBEEAMP, TechwithLucy,tressuni, itgirltierra, DarshilParmar, jayzern




In [ ]:
channel_ids = []

for handle in channel_handles:
    request = youtube.channels().list(
        part="id",
        forHandle=handle
    )
    response = request.execute()

    # Extract and store the channel ID
    if "items" in response and len(response["items"]) > 0:
        channel_id = response["items"][0]["id"]
        channel_ids.append(channel_id)
    else:
        print(f"Channel not found for handle: {handle}")

print("Channel IDs:", channel_ids)

Channel IDs: ['UCF931z8s2EvB67ZIBnLN6gA']


In [ ]:
api_service_name = "youtube"
api_version = "v3"



In [ ]:
def get_channel_stats(youtube, channel_ids):
    all_data = []
    request = youtube.channels().list(
        part="snippet,contentDetails,statistics",
        id=','.join(channel_ids)
    )
    response = request.execute()
    for item in response['items']:
      data = {'channelName': item['snippet']['title'],
                'subscribers': item['statistics']['subscriberCount'],
                'views': item['statistics']['viewCount'],
                'totalVideos': item['statistics']['videoCount'],
                'playlistId': item['contentDetails']['relatedPlaylists']['uploads']
                }

      all_data.append(data)
    return (pd.DataFrame(all_data))


In [ ]:
channel_stats = get_channel_stats(youtube, channel_ids)

In [ ]:
channel_stats

,channelName,subscribers,views,totalVideos,playlistId
0,jayzern,19200,760089,12,UUF931z8s2EvB67ZIBnLN6gA


In [ ]:
import pandas as pd
import time

In [ ]:
def get_video_ids(youtube, playlist_id):
    video_ids = []
    next_page_token = None

    while True:
        request = youtube.playlistItems().list(
            part="contentDetails",
            playlistId=playlist_id,
            maxResults=50,
            pageToken=next_page_token
        )
        response = request.execute()

        for item in response['items']:
            video_ids.append(item['contentDetails']['videoId'])

        next_page_token = response.get('nextPageToken')
        if not next_page_token:
            break

    return video_ids

In [ ]:
playlist_ids = channel_stats['playlistId'].tolist()
all_video_ids = []

for playlist_id in playlist_ids:
    print(f"Fetching videos from playlist: {playlist_id}")
    ids = get_video_ids(youtube, playlist_id)
    all_video_ids.extend(ids)
    print(f"Collected {len(ids)} videos from this playlist.")

print(f"\nTotal video IDs collected: {len(all_video_ids)}")


Fetching videos from playlist: UUF931z8s2EvB67ZIBnLN6gA
Collected 12 videos from this playlist.

Total video IDs collected: 12


In [ ]:
import pandas as pd
import time

def get_video_details(youtube, video_ids, output_csv='video_data.csv'):
    batch_size = 50
    first_write = True  # To write header only once

    for i in range(0, len(video_ids), batch_size):
        batch = video_ids[i:i+batch_size]

        request = youtube.videos().list(
            part="snippet,contentDetails,statistics",
            id=",".join(batch)
        )
        response = request.execute()

        all_video_info = []
        for video in response['items']:
            stats_to_keep = {
                'snippet': ['channelTitle', 'title', 'description', 'tags', 'publishedAt'],
                'statistics': ['viewCount', 'likeCount', 'favoriteCount', 'commentCount'],
                'contentDetails': ['duration', 'definition', 'caption']
            }

            video_info = {'video_id': video['id']}
            for k in stats_to_keep:
                for v in stats_to_keep[k]:
                    try:
                        video_info[v] = video[k][v]
                    except:
                        video_info[v] = None

            all_video_info.append(video_info)

        # Convert to DataFrame and append to CSV
        df = pd.DataFrame(all_video_info)
        df.to_csv(output_csv, mode='a', index=False, header=first_write)
        first_write = False  # After first write, skip writing header

        time.sleep(1)  # Pause to respect quota, optional

    print(f"✅ Done! Data saved to '{output_csv}'")


In [ ]:
get_video_details(youtube, all_video_ids, output_csv='jayzern.csv')

✅ Done! Data saved to 'jayzern.csv'


In [ ]:
# List your files
csv_files = ['5mincrafts.csv', 'Bright_side.csv', 'Buzzfeed.csv', 'Tasty_6000.csv','Thu_vu.csv','averysmith.csv','grateandgarnish5877.csv','thefitnessmarshall.csv','tina_yong.csv','VickyZhaoBEEAMP.csv', 'TechwithLucy.csv','tressuni.csv', 'itgirltierra.csv', 'DarshilParmar.csv',
'jayzern.csv' ]

# Read and merge all into one DataFrame
merged_df = pd.concat([pd.read_csv(file) for file in csv_files], ignore_index=True)

# Save the merged DataFrame to a new CSV
merged_df.to_csv('merged_file.csv', index=False)

print("✅ Successfully merged all CSV files into 'merged_file.csv'")